In [ ]:
#TASK1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)
sns.set_style("whitegrid")

data_a = np.random.normal(loc=50, scale=10, size=500)

data_b = np.random.exponential(scale=15, size=500) + 20 

data_c = np.concatenate([
    np.random.normal(loc=30, scale=5, size=250),
    np.random.normal(loc=70, scale=5, size=250)
])

def calculate_stats(data):
    return {
        "Mean": np.mean(data),
        "Median": np.median(data),
        "Mode": stats.mode(data, keepdims=True)[0][0],
        "Std Dev": np.std(data),
        "Variance": np.var(data),
        "IQR": stats.iqr(data),
        "Skewness": stats.skew(data),
        "Kurtosis": stats.kurtosis(data)
    }


stats_df = pd.DataFrame({
    "Symmetric": calculate_stats(data_a),
    "Right-Skewed": calculate_stats(data_b),
    "Bimodal": calculate_stats(data_c)
}).round(2)

print(stats_df)

### Guiding Question: Which summary statistics look similar across the three datasets, and which reveal the differences?

Similar Statistics: Depending on how the distributions are shifted, the Mean and Median can sometimes look deceptively similar across different shapes. For instance, a Symmetric distribution and a Bimodal distribution can both be centered at 50, yielding nearly identical means.

Differentiating Statistics: * Skewness: This is the primary "whistleblower" for Dataset B. While Dataset A (Symmetric) will have a skewness near 0, Dataset B (Right-Skewed) will have a high positive value.

Kurtosis: This reveals the "tailedness." The Bimodal distribution (Dataset C) often shows negative kurtosis because the data is pushed away from the center toward the two peaks, unlike the bell curve of Dataset A.

Standard Deviation: Even if the means are the same, the spread in the Bimodal dataset is often much higher because the data points are clustered far from the center.

In [ ]:
#TASK2
datasets = [data_a, data_b, data_c]
titles = ["Symmetric", "Right-Skewed", "Bimodal"]

fig, axes = plt.subplots(3, 3, figsize=(18, 15))

for i, data in enumerate(datasets):
    # Histogram + KDE
    sns.histplot(data, bins=30, kde=True, ax=axes[i, 0], color='skyblue')
    axes[i, 0].set_title(f"{titles[i]} - Histogram")
    
    # Boxplot
    sns.boxplot(x=data, ax=axes[i, 1], color='lightgreen')
    axes[i, 1].set_title(f"{titles[i]} - Boxplot")
    
    # KDE with Mean/Median lines
    sns.kdeplot(data, ax=axes[i, 2], fill=True, color='salmon')
    axes[i, 2].axvline(np.mean(data), color='red', linestyle='--', label=f'Mean: {np.mean(data):.2f}')
    axes[i, 2].axvline(np.median(data), color='blue', linestyle='-', label=f'Median: {np.median(data):.2f}')
    axes[i, 2].set_title(f"{titles[i]} - KDE (Mean vs Median)")
    axes[i, 2].legend()

plt.tight_layout()
plt.show()

### Guiding Question: For which dataset(s) do the mean and median diverge the most? What visual feature explains the divergence?

The Divergence: The Mean and Median diverge most significantly in Dataset B (Right-Skewed).

The Visual Feature: This is caused by the "Long Tail" on the right side of the distribution.

The Mean is sensitive to every value in the dataset; therefore, the extreme high values in the tail pull the mean toward the right.

The Median is a positional statistic (the middle point); it remains largely unaffected by the actual value of those extreme points as long as their rank doesn't change.

Bimodal Insight: In Dataset C, the mean might fall in the "valley" between the two peaks—a place where almost no data actually exists—showing that the mean is not always a "typical" value.

In [ ]:
#Creat copy
data_outlier = data_a.copy()

indices = np.argsort(data_outlier)[-5:]
data_outlier[indices] = data_outlier[indices] * 10

outlier_stats = pd.DataFrame({
    "Original": calculate_stats(data_a),
    "With Outliers": calculate_stats(data_outlier)
}).round(2)

print(outlier_stats)

# Side-by-side Boxplot
plt.figure(figsize=(10, 6))
sns.boxplot(data=[data_a, data_outlier], orient='h', showmeans=True, 
            meanprops={"marker":"D","markerfacecolor":"white", "markeredgecolor":"black"})
plt.yticks([0, 1], ['Original', 'With Outliers'])
plt.title("Effect of Outlier on Mean (White Diamond) and Distribution")
plt.show()

### Guiding Question: Which statistics changed the most? Which were robust? If you were reporting a "typical" value, which would you choose?

Highly Sensitive Statistics: The Mean, Variance, and Standard Deviation changed drastically. Because these formulas involve summing all values (and squaring them for variance), even five extreme outliers can inflate the "average" to a level that no longer represents the majority of the group.

Robust Statistics: The Median and IQR (Interquartile Range) remained relatively stable. Since they are based on the ranking of data rather than the magnitude of every value, they "ignore" the extreme nature of the outliers.

Choosing the "Typical" Value: For the modified dataset, I would choose the Median.

Why? The Mean is now "skewed" by the 5 outliers and suggests a center that is much higher than where most of the data points actually sit. The Median provides a more honest representation of the "middle" of the bulk of the data.